# DQC End-to-End Workflow

Runs the full DQC pipeline:

1. **Configure** — choose circuit mode (`cisco` / `tket`), file, and measurement qubits
2. **Send** the circuit to the QNCP DQC plugin via RPC
3. **Simulate** from the labeled payload returned by the plugin
4. **Inspect** bitstring results and save CSV

Prerequisites: `qnpack` installed in the Python environment (`pip install -e /path/to/qnpack`).

## 0. Configuration

In [1]:
import os

CONTROLLER_HOST = os.getenv("HOST", "localhost")
NUM_RUNS        = 10

# Paths relative to this notebook's directory
_NB_DIR     = os.path.dirname(os.path.abspath("__file__"))
SCHEMA_PATH = os.path.join(_NB_DIR, "schema", "dqc.yaml")
PARAMS      = os.path.join(_NB_DIR, "parameters.yml")

# ── Circuit selection ─────────────────────────────────────────────────────────
# Set CIRCUIT_MODE to "cisco" or "tket".
# For "tket":  set CIRCUIT_FILE to a commands/*.txt path and MEASURE_QUBITS.
# For "cisco": set CIRCUIT_FILE to a qasm/*.qasm path (MEASURE_QUBITS not needed).

CIRCUIT_MODE   = "tket"   # "cisco" or "tket"

# Tket examples (dist_commands + measure_qubits):
CIRCUIT_FILE   = os.path.join(_NB_DIR, "commands", "grover_4_2qpu.txt")
MEASURE_QUBITS = "{2: [21, 20], 1: [21, 20]}"   # set None to skip DataCollector path

# Cisco example:
# CIRCUIT_MODE   = "cisco"
# CIRCUIT_FILE   = os.path.join(_NB_DIR, "qasm", "grover4_2qpu.qasm")
# MEASURE_QUBITS = None

print(f"Controller:     {CONTROLLER_HOST}")
print(f"Schema:         {SCHEMA_PATH}  (exists={os.path.exists(SCHEMA_PATH)})")
print(f"Parameters:     {PARAMS}  (exists={os.path.exists(PARAMS)})")
print(f"Circuit mode:   {CIRCUIT_MODE}")
print(f"Circuit file:   {CIRCUIT_FILE}  (exists={os.path.exists(CIRCUIT_FILE)})")
print(f"Measure qubits: {MEASURE_QUBITS}")
print(f"Num runs:       {NUM_RUNS}")

Controller:     localhost
Schema:         /home/esnet/work/qnpack-tutorials/examples/DQC_examples/schema/dqc.yaml  (exists=False)
Parameters:     /home/esnet/work/qnpack-tutorials/examples/DQC_examples/parameters.yml  (exists=False)
Circuit mode:   tket
Circuit file:   /home/esnet/work/qnpack-tutorials/examples/DQC_examples/commands/grover_4_2qpu.txt  (exists=False)
Measure qubits: {2: [21, 20], 1: [21, 20]}
Num runs:       10


## 1. Run via QNCP Plugin + Simulate

In [2]:
from qnpack.dqc.sim import DQCSimulation
from quantnet_mq.schema.models import Schema

Schema.load_schema(SCHEMA_PATH, ns="dqc")

circuit_cfg = {
    "mode": CIRCUIT_MODE,
    "pre_schedule_entanglement": False,
    ("dist_commands_file" if CIRCUIT_MODE == "tket" else "qasm_file"): CIRCUIT_FILE,
}
if MEASURE_QUBITS is not None:
    circuit_cfg["measure_qubits"] = MEASURE_QUBITS

sim = DQCSimulation(
    parameter_file=PARAMS,
    fixed_params={"circuit": circuit_cfg},
    varying_params={},
)

print(f"Sending circuit to plugin at {CONTROLLER_HOST} ...")
sim_results = sim.start(num_runs=NUM_RUNS, qncp_host=CONTROLLER_HOST)

print(f"Completed {len(sim_results)} run(s).")

ModuleNotFoundError: No module named 'quantnet_mq'

## 2. Inspect Results

In [ ]:
import pandas as pd
from collections import Counter

df = pd.DataFrame(sim_results)

# Detect measurement columns (cisco → m_N; tket → QPU_N_qP)
meas_cols = [c for c in df.columns if c.startswith("m_") or c.startswith("QPU_")]

print(f"{len(sim_results)} simulation run(s) completed")
print(f"Measurement columns: {meas_cols}")

bitstrings = df["bitstring"].tolist()
counts = Counter(bitstrings)
print(f"\nTop bitstrings:")
for bs, cnt in counts.most_common(10):
    pct = 100 * cnt / len(sim_results)
    print(f"  {bs}  ({cnt}/{len(sim_results)}, {pct:.1f}%)")

display_cols = ["run", "bitstring", "sim_duration_s"] + meas_cols
print(f"\nResults (first 10 rows):")
print(df[display_cols].head(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

labels = sorted(counts.keys())
values = [counts[k] for k in labels]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.6), 4))
ax.bar(labels, values, color="steelblue", edgecolor="black")
ax.set_xlabel("Bitstring")
ax.set_ylabel("Count")
ax.set_title(f"Bitstring distribution — {NUM_RUNS} runs ({CIRCUIT_MODE}: {os.path.basename(CIRCUIT_FILE)})")
ax.tick_params(axis='x', rotation=45, labelsize=8)
for i, v in enumerate(values):
    ax.text(i, v + 0.2, str(v), ha="center", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Save CSV

In [ ]:
circuit_stem = os.path.splitext(os.path.basename(CIRCUIT_FILE))[0]
csv_name = f"results_{circuit_stem}_{NUM_RUNS}runs.csv"
CSV_OUT  = os.path.join(_NB_DIR, "results", csv_name)
os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)

df.to_csv(CSV_OUT, index=False)
print(f"Saved {len(df)} row(s) → {CSV_OUT}")
print(f"Columns: {list(df.columns)}")